# 第二章（一）：单旋律符号表示

以一条短旋律为例，比较同一段音乐的四种计算表示：

1. **MIDI 音符对象**：由 SMF 起音与释音消息配对得到的音符记录
2. **音高/时长序列**：不含起始位置、休止与力度的紧凑表示
3. **事件标记（event token）**：包含起音、释音、时间推进与力度的离散序列
4. **钢琴卷帘（piano roll）**：时间--音高网格；本例数组存储为音高 $\times$ 时间

数据：`CODE/datasets/melodies/红河谷.midi`
图片输出：`CODE/chapter02/output_figures/`（600 dpi）

In [ ]:
import math
import os
import warnings

import matplotlib.pyplot as plt
import mido
import pretty_midi

warnings.filterwarnings("ignore", message="Tempo, Key or Time signature change events found.*", category=RuntimeWarning)

# 中文字体
plt.rcParams["font.sans-serif"] = [
    "PingFang SC", "Hiragino Sans GB", "Microsoft YaHei",
    "SimHei", "Arial Unicode MS", "Noto Sans CJK SC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

# 强制白底
for _k in ("figure.facecolor", "axes.facecolor"):
    plt.rcParams[_k] = "white"
for _k in ("axes.edgecolor", "axes.labelcolor", "xtick.color", "ytick.color", "text.color"):
    plt.rcParams[_k] = "black"

# 路径
_p = os.getcwd()
while not os.path.exists(os.path.join(_p, "CODE", "datasets")):
    _parent = os.path.dirname(_p)
    if _parent == _p:
        raise FileNotFoundError("未找到包含 CODE/datasets 的项目根目录")
    _p = _parent
BASE_DIR = _p

MIDI_PATH = os.path.join(BASE_DIR, "CODE", "datasets", "melodies", "红河谷.midi")
FIGURES_DIR = os.path.join(BASE_DIR, "CODE", "chapter02", "output_figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("MIDI_PATH:", MIDI_PATH)

## 1. MIDI 音符事件

标准 MIDI 文件（SMF）保存带时间信息的演奏消息和元事件，不保存音频波形。代码分别读取 SMF 轨道和 `pretty_midi` 解析后的音符对象。

In [ ]:
smf = mido.MidiFile(MIDI_PATH)
pm = pretty_midi.PrettyMIDI(MIDI_PATH)

print(f"SMF 轨道数：{len(smf.tracks)}")
print(f"pretty_midi Instrument 对象数：{len(pm.instruments)}")
print(f"拍号：{pm.time_signature_changes}")
print(f"速度：{pm.get_tempo_changes()[1]} BPM")
print()

# 提取音符事件
instrument = pm.instruments[0]
notes = sorted(instrument.notes, key=lambda n: (n.start, n.pitch, n.end))
print(f"音符总数：{len(notes)}")
print(f"音高范围：MIDI {min(n.pitch for n in notes)}～{max(n.pitch for n in notes)}")
print(f"末个音符终止时间：{notes[-1].end:.2f} 秒")

In [ ]:
# 音符事件表
header = f"{'序号':>3}  {'音名':<5} {'MIDI':>4}  {'起始(s)':>7} {'终止(s)':>7} {'时长(s)':>7} {'力度':>4}"
print(header)
print("-" * len(header))
for i, event_note in enumerate(notes[:12], start=1):
    event_note_name = pretty_midi.note_number_to_name(event_note.pitch)
    event_duration_seconds = event_note.end - event_note.start
    print(f"{i:>3}  {event_note_name:<5} {event_note.pitch:>4}  {event_note.start:>7.3f} {event_note.end:>7.3f} {event_duration_seconds:>7.3f} {event_note.velocity:>4}")

### 读取前五个音符对象

输出中的每一行给出 `pretty_midi.Note` 对象的起始时间、时长和触发力度；对象的终止时间不包含踏板造成的延续。

In [ ]:
for preview_note in notes[:5]:
    preview_note_name = pretty_midi.note_number_to_name(preview_note.pitch)
    preview_duration_seconds = preview_note.end - preview_note.start
    print(f"{preview_note.start:.2f} 秒：起音 {preview_note_name}（MIDI 音高 {preview_note.pitch}），"
          f"触发力度 {preview_note.velocity}，Note 对象时长 {preview_duration_seconds:.2f} 秒")

## 2. 音高/时长序列

提取每个音符的音高与时长并组成二元组，会丢弃力度、起始位置与休止信息。这种表示可用于不需要上述信息的单旋律分析或序列模型。

In [ ]:
pitch_seq = [n.pitch for n in notes]
dur_seq = [round(float(n.end - n.start), 3) for n in notes]
name_seq = [pretty_midi.note_number_to_name(n.pitch) for n in notes]

print("音高序列 (MIDI 编号):")
print(pitch_seq[:20], "..." if len(pitch_seq) > 20 else "")
print()
print("音高序列 (音名):")
print(name_seq[:20], "..." if len(name_seq) > 20 else "")
print()
print("时长序列（秒）：")
print(dur_seq[:20], "..." if len(dur_seq) > 20 else "")
print()
print("（音高，时长）对：")
pairs = [(name_seq[i], dur_seq[i]) for i in range(min(10, len(notes)))]
for p in pairs:
    print(f"  {p}")

## 3. 事件标记

本例使用四类事件标记：`note_on` 表示起音，`note_off` 表示释音，`time_shift` 推进时间，`velocity` 设置后续起音的力度。这套自定义编码使用小写名称、十进制秒数和原始力度值，与 Music Transformer 论文的词表不同。事件时间先量化到 0.01 秒；同一时刻先处理释音，再处理起音。在同一音高不发生重叠的条件下，该序列可恢复量化后的起始位置、时长和力度。

In [ ]:
def notes_to_event_tokens(note_list, time_decimals=2):
    """编码 note_on、note_off、time_shift 与 velocity 事件。"""
    raw_events = []
    for token_note in note_list:
        start = float(token_note.start)
        end = float(token_note.end)
        if not math.isfinite(start) or not math.isfinite(end):
            raise ValueError(f"音符 {token_note.pitch} 的起止时间必须为有限值")
        pitch = int(token_note.pitch)
        velocity = int(token_note.velocity)
        if pitch != token_note.pitch or not 0 <= pitch <= 127:
            raise ValueError("音高必须是 0 至 127 之间的整数")
        if velocity != token_note.velocity or not 1 <= velocity <= 127:
            raise ValueError("触发力度必须是 1 至 127 之间的整数")
        if start < 0:
            raise ValueError(f"音符 {token_note.pitch} 的起始时间不能为负数")
        quantized_start = round(start, time_decimals)
        quantized_end = round(end, time_decimals)
        if quantized_end <= quantized_start:
            raise ValueError(f"音符 {token_note.pitch} 量化后的时长必须大于 0")
        raw_events.append((quantized_start, 1, "note_on", pitch, velocity))
        raw_events.append((quantized_end, 0, "note_off", pitch, None))
    raw_events.sort(key=lambda event: (event[0], event[1], event[3]))

    event_tokens = []
    current_time = 0.0
    current_velocity = None
    active_pitches = set()
    for event_time, _, event_type, pitch, velocity in raw_events:
        time_shift = round(event_time - current_time, time_decimals)
        if time_shift > 0:
            event_tokens.append(f"time_shift_{time_shift:.{time_decimals}f}")
        current_time = event_time
        if event_type == "note_on":
            if pitch in active_pitches:
                raise ValueError(f"本例不支持同一音高重叠：{pitch}")
            active_pitches.add(pitch)
            if velocity != current_velocity:
                event_tokens.append(f"velocity_{velocity}")
                current_velocity = velocity
            event_tokens.append(f"note_on_{pitch}")
        else:
            if pitch not in active_pitches:
                raise ValueError(f"音符 {pitch} 的释音没有对应起音")
            active_pitches.remove(pitch)
            event_tokens.append(f"note_off_{pitch}")
    return event_tokens


def event_tokens_to_notes(tokens, time_decimals=2):
    """解码本例标记；同一音高在任一时刻最多保留一个活动音符。"""
    current_time = 0.0
    current_velocity = None
    active_notes = {}
    decoded_notes = []

    for token in tokens:
        event_type, value = token.rsplit("_", 1)
        if event_type == "time_shift":
            shift = float(value)
            if not math.isfinite(shift) or shift <= 0:
                raise ValueError("time_shift 必须是大于 0 的有限值")
            next_time = round(current_time + shift, time_decimals)
            if next_time <= current_time:
                raise ValueError("time_shift 在当前量化精度下必须推进时间")
            current_time = next_time
        elif event_type == "velocity":
            current_velocity = int(value)
            if not 1 <= current_velocity <= 127:
                raise ValueError("velocity 必须在 1 至 127 之间")
        elif event_type == "note_on":
            pitch = int(value)
            if not 0 <= pitch <= 127:
                raise ValueError("note_on 音高必须在 0 至 127 之间")
            if current_velocity is None:
                raise ValueError("note_on 之前缺少 velocity 标记")
            if pitch in active_notes:
                raise ValueError(f"本例不支持同一音高重叠：{pitch}")
            active_notes[pitch] = (current_time, current_velocity)
        elif event_type == "note_off":
            pitch = int(value)
            if not 0 <= pitch <= 127:
                raise ValueError("note_off 音高必须在 0 至 127 之间")
            if pitch not in active_notes:
                raise ValueError(f"note_off 没有对应的活动音符：{pitch}")
            start, velocity = active_notes.pop(pitch)
            if current_time <= start:
                raise ValueError(f"音符 {pitch} 的解码时长必须大于 0")
            decoded_notes.append((pitch, start, current_time, velocity))
        else:
            raise ValueError(f"未知事件标记：{token}")

    if active_notes:
        raise ValueError(f"序列结束时仍有活动音符：{sorted(active_notes)}")
    return sorted(decoded_notes, key=lambda item: (item[1], item[0], item[2]))


event_tokens = notes_to_event_tokens(notes)
decoded_notes = event_tokens_to_notes(event_tokens)
expected_notes = sorted(
    [
        (note.pitch, round(float(note.start), 2), round(float(note.end), 2), note.velocity)
        for note in notes
    ],
    key=lambda item: (item[1], item[0], item[2]),
)
assert decoded_notes == expected_notes

def expect_value_error(function, *args):
    try:
        function(*args)
    except ValueError:
        return
    raise AssertionError("预期 ValueError，但函数正常返回")

expect_value_error(
    notes_to_event_tokens,
    [pretty_midi.Note(90, 60, 0.0, 1.0), pretty_midi.Note(90, 60, 0.5, 1.5)],
)
expect_value_error(notes_to_event_tokens, [pretty_midi.Note(90, 60, -0.004, 0.5)])
expect_value_error(notes_to_event_tokens, [pretty_midi.Note(90, 60, 0.0, 0.004)])
for invalid_tokens in (
    ["note_on_60"],
    ["velocity_90", "note_off_60"],
    ["velocity_90", "note_on_60"],
    ["velocity_90", "note_on_60", "note_off_60"],
    ["time_shift_-0.10"],
    ["time_shift_nan"],
    ["time_shift_0.004"],
):
    expect_value_error(event_tokens_to_notes, invalid_tokens)
print(f"事件标记总数：{len(event_tokens)}")
print(f"本序列出现的不同标记数：{len(set(event_tokens))}")
print(f"解码校验：{len(decoded_notes)} 个音符对象与 0.01 秒量化结果一致")
print("边界检查：非有限时间、量化或解码零时长、同音高重叠和所测无效事件序列均被拒绝")
print("前 24 个事件标记：")
for i, token in enumerate(event_tokens[:24]):
    print(f"  [{i:>2}] {token}")

In [ ]:
# 词表一览
vocab = sorted(set(event_tokens))
print(f"本序列出现的标记（{len(vocab)} 个）：")
for i, token in enumerate(vocab):
    print(f"  {token}", end="    " if (i + 1) % 5 else "\n")
if len(vocab) % 5:
    print()

## 4. 钢琴卷帘（piano roll）

钢琴卷帘按 0.01 秒分辨率生成。图中横轴为时间、纵轴为 MIDI 音高；`pretty_midi.get_piano_roll()` 返回的数组形状为 `(128, 时间步数)`，即第一轴是音高、第二轴是时间。矩阵中的非零格表示相应音高在该时间步处于活动状态。本例没有延音踏板消息，因此未额外计入踏板延音。

In [ ]:
assert not any(cc.number == 64 for cc in instrument.control_changes)
roll_frames_per_second = 100
piano_roll = instrument.get_piano_roll(
    fs=roll_frames_per_second,
    pedal_threshold=None,
)
active_roll = piano_roll > 0
print(f"钢琴卷帘矩阵形状：{active_roll.shape}（音高 × 时间步）")
print(f"时间分辨率：{1 / roll_frames_per_second:.2f} 秒")
figure_title = "《红河谷》钢琴卷帘"
print(f"图：{figure_title}")

pitch_min = min(n.pitch for n in notes) - 2
pitch_max = max(n.pitch for n in notes) + 2
roll_duration_seconds = active_roll.shape[1] / roll_frames_per_second
fig, ax = plt.subplots(figsize=(12, 4))
ax.imshow(
    active_roll[pitch_min:pitch_max + 1],
    origin="lower",
    aspect="auto",
    interpolation="nearest",
    cmap="Greys",
    vmin=0,
    vmax=1,
    extent=[0, roll_duration_seconds, pitch_min - 0.5, pitch_max + 0.5],
)
ax.set_xlim(0, notes[-1].end + 0.5)
ax.set_ylim(pitch_min, pitch_max)

# 纵轴标注音名
yticks = list(range(pitch_min + 1, pitch_max))
ax.set_yticks(yticks)
ax.set_yticklabels([pretty_midi.note_number_to_name(p) for p in yticks], fontsize=8)

ax.set_xlabel("时间（秒）")
ax.set_ylabel("MIDI 音高")
ax.grid(axis="y", linestyle=":", alpha=0.3)

plt.tight_layout()
out_path = os.path.join(FIGURES_DIR, "piano_roll_melody.png")
fig.savefig(out_path, dpi=600, bbox_inches="tight")
plt.show()
print(f"已保存：{out_path}")

## 汇总

| 表示           | 形式                         | 特点                 |
| -------------- | ---------------------------- | -------------------- |
| MIDI 音符对象 | `(pitch, start, end, velocity)` | 保留起始、终止与触发力度；不等同于全部 SMF 消息 |
| 音高/时长序列 | `(pitch, duration)` | 紧凑，但丢弃起始位置、休止和力度 |
| 事件标记 | 离散字符串序列 | 本例可恢复量化后的起止时间与力度 |
| 钢琴卷帘 | 二维矩阵（本例为音高 $\times$ 时间） | 显示音高活动状态，可按模型接口转置 |

表示选择应依据任务所需信息：需要保留播放控制与轨道结构时，应读取 SMF 消息；不需要同时性、力度、绝对起始位置和休止信息的旋律模型可使用音高/时长序列；序列模型可使用定义明确的事件标记；矩阵模型或可视化可使用钢琴卷帘。

| 输出文件                     | 内容             |
| ---------------------------- | ---------------- |
| `piano_roll_melody.png` | 《红河谷》钢琴卷帘 |